# 01. Exploratory Data Analysis (EDA) & Data Pipeline

**Dự án**: `industrial-defect-detection-onnx`  
**Mục tiêu**: Kiểm tra và phân tích chất lượng dữ liệu, đánh giá tỷ lệ mất cân bằng (Imbalance Ratio), trực quan hóa pipeline augmentation chuyên biệt cho defect detection (CLAHE, ColorJitter, Flips).

In [ ]:
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add root project to sys.path
sys.path.append("..")
from src.dataset import DefectDataset, get_transforms, create_dataloaders, get_class_imbalance_info
from src.dataset_eda import analyze_dataset, plot_eda_distribution, plot_eda_samples

## 1. Phân Tích Tổng Quan Phân Bố Tập Dữ Liệu

In [ ]:
data_dir = "../data/processed"
stats = analyze_dataset(data_dir)

for split, info in stats.items():
    imb = info["imbalance_info"]
    print(f"Split: {split.upper():<5} | Good: {imb['num_good']:<4} | Defect: {imb['num_defect']:<4} | Defect Ratio: {imb['defect_percentage']:.1f}%")

## 2. Trực Quan Hóa Augmentation Pipeline (CLAHE, Flips, ColorJitter)

In [ ]:
train_transform = get_transforms(img_size=(224, 224), is_train=True)
train_ds = DefectDataset.from_directory(f"{data_dir}/train", transform=train_transform)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    # Sample good
    img_t, lbl = train_ds[i]
    img_np = img_t.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np * (0.229, 0.224, 0.225) + (0.485, 0.456, 0.406), 0, 1)
    axes[0, i].imshow(img_np)
    axes[0, i].set_title(f"Train Good (Lbl={lbl.item()})")
    axes[0, i].axis("off")

    # Sample defect (from last elements)
    img_t_def, lbl_def = train_ds[len(train_ds) - 1 - i]
    img_np_def = img_t_def.permute(1, 2, 0).numpy()
    img_np_def = np.clip(img_np_def * (0.229, 0.224, 0.225) + (0.485, 0.456, 0.406), 0, 1)
    axes[1, i].imshow(img_np_def)
    axes[1, i].set_title(f"Train Defect (Lbl={lbl_def.item()})")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## 3. Kiểm Tra WeightedRandomSampler Trong Batch Train

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(data_dir, batch_size=16, num_workers=0)
batch_images, batch_labels = next(iter(train_loader))

print(f"Batch Images Shape: {batch_images.shape}")
print(f"Batch Labels Shape: {batch_labels.shape}")
print(f"Defect Count in Sampled Batch: {(batch_labels == 1).sum().item()} / {len(batch_labels)}")
print(f"Good Count in Sampled Batch: {(batch_labels == 0).sum().item()} / {len(batch_labels)}")